In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import TargetEncoder


In [ ]:
df = pd.read_csv("Divar.csv")
df_ml = df.copy()

In [ ]:
df_ml = df.copy()


def clean_money(series):
    s = series.astype(str).str.translate(
        str.maketrans(
            '۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩',
            '01234567890123456789'
        )
    )

    s = s.str.replace(r'[^\d.]', '', regex=True)

    return pd.to_numeric(s, errors='coerce')


df_ml['price_clean'] = clean_money(df_ml['price_value'])
df_ml['credit_clean'] = clean_money(df_ml['credit_value'])
df_ml['rent_clean'] = clean_money(df_ml['rent_value'])


RENT_TO_CREDIT = 33.33
CREDIT_TO_PRICE = 10

df_ml['total_credit'] = (
    df_ml['credit_clean'].fillna(0)
    + df_ml['rent_clean'].fillna(0) * RENT_TO_CREDIT
)

df_ml['estimated_price'] = (
    df_ml['total_credit'] * CREDIT_TO_PRICE
)

df_ml['target_price'] = (
    df_ml['price_clean'].fillna(
        df_ml['estimated_price']
    )
)


print("تعداد اولیه داده‌ها:", len(df_ml))
print("تعداد target معتبر:", (df_ml['target_price'] > 0).sum())

تعداد اولیه داده‌ها: 1000000
تعداد target معتبر: 918647


<div dir="rtl" style="text-align:right; font-family:'Segoe UI','B Nazanin',Arial,sans-serif; font-size:15px; line-height:2;">

<h3>آماده‌سازی داده‌ها و ساخت متغیر هدف</h3>

<p>
در این مرحله ابتدا یک کپی از داده‌های اصلی ایجاد کردیم تا تغییرات مربوط به مدل‌سازی روی داده‌های اصلی اعمال نشود.
</p>

<p>
از آنجا که مقادیر مربوط به قیمت، رهن و اجاره ممکن است به صورت متنی و با ارقام فارسی یا جداکننده‌های مختلف ذخیره شده باشند، ابتدا آن‌ها را به مقادیر عددی تبدیل کردیم.
</p>

<p>
برای آگهی‌های فروش، مقدار <b>price_value</b> به عنوان قیمت هدف استفاده شد.
برای آگهی‌های اجاره، ابتدا مقدار رهن و اجاره را به یک ارزش اعتباری تقریبی تبدیل کرده و سپس آن را به قیمت تقریبی کل ملک تبدیل کردیم.
</p>

<p>
در نتیجه، ستون <b>target_price</b> به عنوان متغیر هدف مدل ساخته شد که نشان‌دهنده ارزش تقریبی ملک است.
</p>

</div>

In [ ]:
df_ml = df_ml[
    df_ml['target_price'] > 500_000_000
].copy()


upper_bound = df_ml['target_price'].quantile(0.99)

df_ml = df_ml[
    df_ml['target_price'] <= upper_bound
].copy()


numeric_features = [
    'building_size',
    'rooms_count',
    'construction_year',
    'floor',
    'total_floors_count',
    'unit_per_floor',
    'is_sale',
    'has_elevator',
    'has_parking',
    'has_warehouse'
]


for col in numeric_features:
    if col in df_ml.columns:
        df_ml[col] = pd.to_numeric(
            df_ml[col],
            errors='coerce'
        )


df_ml['is_sale'] = (
    df_ml['price_clean'].notna()
).astype(int)


bool_features = [
    'has_elevator',
    'has_parking',
    'has_warehouse'
]

for col in bool_features:
    df_ml[col] = df_ml[col].apply(
        lambda x: 1
        if str(x).strip().lower()
        in ['true', '1', 'دارد', 'yes']
        else 0
    )


categorical_features = [
    'city_slug',
    'neighborhood_slug',
    'property_type',
    'transaction_type'
]


df_ml['transaction_type'] = np.where(
    df_ml['price_clean'].notna(),
    'sale',
    np.where(
        (df_ml['credit_clean'] > 0) &
        (df_ml['rent_clean'] > 0),
        'rent_credit',
        np.where(
            df_ml['credit_clean'] > 0,
            'full_credit',
            'rent'
        )
    )
)


X = df_ml[
    numeric_features + categorical_features
].copy()

y = df_ml['target_price'].copy()


print("تعداد داده‌های نهایی:", len(df_ml))
print("\nانواع معاملات:")
print(df_ml['transaction_type'].value_counts())

تعداد داده‌های نهایی: 855924

انواع معاملات:
transaction_type
sale           516400
rent_credit    279034
full_credit     58689
rent             1801
Name: count, dtype: int64


<div dir="rtl" style="text-align:right; font-family:'Segoe UI','B Nazanin',Arial,sans-serif; font-size:15px; line-height:2;">

<h3>پاک‌سازی داده‌ها و انتخاب ویژگی‌ها</h3>

<p>
در این مرحله ابتدا آگهی‌هایی که مقدار هدف آن‌ها کمتر از ۵۰۰ میلیون تومان بود حذف شدند تا داده‌های بسیار کوچک و نامعتبر روی مدل اثر زیادی نداشته باشند.
</p>

<p>
همچنین برای کاهش تأثیر قیمت‌های بسیار بزرگ و غیرعادی، یک درصد بالاترین مقادیر <b>target_price</b> حذف شد.
</p>

<p>
سپس ویژگی‌های عددی مانند متراژ ساختمان، تعداد اتاق‌ها، سال ساخت، طبقه و تعداد طبقات به نوع عددی تبدیل شدند.
</p>

<p>
ویژگی‌های مربوط به امکاناتی مانند آسانسور، پارکینگ و انباری نیز به صورت صفر و یک نمایش داده شدند.
</p>

<p>
در ادامه، نوع معامله به چهار گروه فروش، رهن و اجاره، رهن کامل و اجاره تقسیم شد و در کنار شهر، محله و نوع ملک به عنوان ویژگی دسته‌ای وارد مدل شد.
</p>

</div>

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)


numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])


categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', TargetEncoder(
        target_type='continuous',
        smooth='auto',
        random_state=42
    ))
])


preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])


model = Pipeline([
    ('preprocessor', preprocessor),

    ('rf', RandomForestRegressor(
        n_estimators=100,
        max_depth=16,
        min_samples_split=10,
        random_state=42,
        n_jobs=-1
    ))
])


print("Train:", X_train.shape)
print("Validation:", X_valid.shape)
print("Test:", X_test.shape)

Train: (599146, 14)
Validation: (128389, 14)
Test: (128389, 14)


<div dir="rtl" style="text-align:right; font-family:'Segoe UI','B Nazanin',Arial,sans-serif; font-size:15px; line-height:2;">

<h3>تقسیم داده‌ها و آماده‌سازی ویژگی‌ها</h3>

<p>
داده‌ها به سه بخش تقسیم شدند: ۷۰ درصد برای آموزش، ۱۵ درصد برای اعتبارسنجی و ۱۵ درصد برای آزمون نهایی.
</p>

<p>
برای ویژگی‌های عددی، مقادیر گمشده با میانه جایگزین شدند.
برای ویژگی‌های دسته‌ای نیز مقادیر گمشده با پرتکرارترین مقدار جایگزین و سپس با روش <b>Target Encoding</b> به مقدار عددی تبدیل شدند.
</p>

<p>
تمام این مراحل داخل یک <b>Pipeline</b> قرار گرفتند تا پردازش داده‌ها به صورت منظم و یکسان روی داده‌های آموزش، اعتبارسنجی و آزمون انجام شود.
</p>

</div>

In [ ]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)


model.fit(
    X_train,
    y_train
)


y_valid_pred = model.predict(X_valid)

print("Validation Results")
print("------------------")

print(
    "R2:",
    r2_score(y_valid, y_valid_pred)
)

print(
    "MAE:",
    mean_absolute_error(y_valid, y_valid_pred)
)

print(
    "RMSE:",
    np.sqrt(
        mean_squared_error(y_valid, y_valid_pred)
    )
)


y_test_pred = model.predict(X_test)

print("\nTest Results")
print("------------")

print(
    "R2:",
    r2_score(y_test, y_test_pred)
)

print(
    "MAE:",
    mean_absolute_error(y_test, y_test_pred)
)

print(
    "RMSE:",
    np.sqrt(
        mean_squared_error(y_test, y_test_pred)
    )
)

c:\Users\ASUS\anaconda3\envs\quera\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['rooms_count' 'construction_year']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\ASUS\anaconda3\envs\quera\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
c:\Users\ASUS\anaconda3\envs\quera\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['rooms_count' 'construction_year']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Validation Results
------------------
R2: 0.6122469028165108
MAE: 2817426727.1819167
RMSE: 6467104380.633054


c:\Users\ASUS\anaconda3\envs\quera\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['rooms_count' 'construction_year']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(



Test Results
------------
R2: 0.6188600126642523
MAE: 2818697236.145988
RMSE: 6463548598.658905


<div dir="rtl" style="text-align:right; font-family:'Segoe UI','B Nazanin',Arial,sans-serif; font-size:15px; line-height:2;">

<h3>آموزش و ارزیابی مدل</h3>

<p>
در این مرحله مدل <b>Random Forest Regressor</b> با استفاده از داده‌های آموزشی آموزش داده شد و سپس روی داده‌های Validation و Test ارزیابی شد.
</p>

<p>
معیار اصلی ارزیابی ما <b>R²</b> است. این معیار نشان می‌دهد مدل چه مقدار از تغییرات موجود در قیمت ملک را می‌تواند با استفاده از ویژگی‌های موجود توضیح دهد.
</p>

<p>
در اجرای انجام‌شده، مقدار R² روی داده‌های Validation حدود <b>0.61</b> و روی داده‌های Test نیز حدود <b>0.61</b> به دست آمد.
</p>

<p>
نزدیک بودن عملکرد Validation و Test نشان می‌دهد که عملکرد مدل روی داده‌های دیده‌نشده نیز تقریباً مشابه است و اختلاف زیادی بین این دو مجموعه وجود ندارد.
</p>

<p>
بنابراین در این مرحله مدل توانسته است حدود ۶۱ درصد از تغییرات قیمت هدف را توضیح دهد.
</p>

</div>